# 8.5. Batch Normalization
D2L의 Batch Normalization장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

## 1. Batch Normalization은 왜 필요한가?

딥러닝에서는 입력 데이터를 학습 전에 정규화하는 경우가 많다. 예를 들어 어떤 feature 값이

```text
키: 150 ~ 190 
몸무게: 40 ~ 120 
재산: 0 ~ 1,000,000
```

처럼 크기가 서로 다르면 학습이 어려워질 수 있다.

그래서 보통 입력 데이터를 평균 ≈ 0 분산 ≈ 1 정도로 맞춘다. 그런데 깊은 신경망에서는 입력 데이터뿐 아니라 중간 layer의 출력값도 학습하며 계속 변한다.

```text
입력
 ↓
Layer 1 → 출력 분포 변화
 ↓
Layer 2 → 출력 분포 변화
 ↓
Layer 3 → 출력 분포 변화
 ↓
...
```

Batch Normalization은 이런 중간값들을 일정한 범위로 정규화해서 학습을 더 안정적으로 만드는 방법이다.

## 2. Batch Normalization의 기본 아이디어

한 minibatch가 있다고 할때

x₁ x₂ x₃ ... xₙ

먼저 minibatch의 평균, 분산을 구한다.

$$
\frac{1}{|B|}
\sum_{x \in B} x
$$
$$
\frac{1}{|B|}
\sum_{x \in B}
(x-\hat{\mu}_B)^2
$$

여기서 $\epsilon$은 0으로 나누는 것을 방지하기 위한 매우 작은 값이다. 그리고 정규화 한다.

$$
\frac{x-\hat{\mu}_B}
{\sqrt{\hat{\sigma}_B^2 + \epsilon}}
$$

대략 평균 -> 0 분산 -> 1이 된다.

## 3. 평균 0, 분산 1로 고정해도 괜찮은가?

신경망이 학습하다 보면 평균이 0이 아닌게 더 좋다거나 값의 크기가 더 큰게 좋은 상황이 있을 수 있다. 그래서 Batch Normalization에는 두 개의 학습 가능한 parameter가 존재한다.

$\gamma$ 와 $\beta$ 이다.

$$
\gamma
\frac{x-\hat{\mu}_B}
{\sqrt{\hat{\sigma}_B^2 + \epsilon}}
+
\beta
$$

$\gamma$ -> 크기조절  
$\beta$ -> 위치 이동

단순히 정규화만 하는 게 아니라 

```text
원래 값
 ↓
정규화
 ↓
평균 0, 분산 1
 ↓
γ로 scale 조절
 ↓
β로 위치 조절
 ↓
최종 출력
```

그리고 $\gamma$, $\beta$는 weight와 마찬가지로 역전파를 통해 학습된다.

## 4. 간단한 숫자 예시

batch가 [2, 4, 6, 8] 일때 평균은 5이다. 각 값에서 평균을 빼면 [-3, -1, 1, 3] 이 된다.

이 값을 표준편차로 나누면 값들이 일정한 범위로 정리된다.

예를 들어서 정규화 결과가 대략 [-1.34, -0.45, 0.45, 1.34]가 되었다고 해보자. γ = 2 β = 1 라면

$$
y = 2\hat{x}+1
$$

BatchNorm은 값의 분포를 한번 정리하고 -> 모델이 원하는 형태로 다시 조절하는 layer라고 생각하면 된다.

## 5. BatchNorm은 어디에 들어갈까

전형적인 사용법은 이렇다.

Conv -> BatchNorm -> Activation

```py
nn.Conv2d(3, 64, kernel_size=3, padding=1), 
nn.BatchNorm2d(64), 
nn.ReLU()
```

Fully Connected Layer에서도 비슷하다.

Linear -> BatchNorm -> Activation

수식으로는 이렇다.
$$
h =\phi(\mathrm{BN}(Wx+b))
$$

Wx + b -> BatchNorm -> ReLU

순서라고 이해하면 된다.

## 6. CNN에서 BatchNorm은 어떻게 계산되나?

CNN의 tensor가 [N, C, H, W] 형태라고 해보자.

예를 들어서 [32, 64, 28, 28]일때

CNN의 BatchNorm은 채널마다 따로 정규화한다. 64개의 채널이 있다면

```text
Channel 1 → 평균/분산 계산
Channel 2 → 평균/분산 계산
Channel 3 → 평균/분산 계산
...
Channel 64 → 평균/분산 계산
```

한 채널에서는 Batch x Height x Width 전체 값을 사용한다. 위 예시에선 하나의 채널당 32 x 28 x 28 개의 값을 이용해 평균과 분산을 계산한다. 따라서 각 채널마다 $\gamma$ 와 $\beta$ 하나씩 존재한다.

## 7. BatchNorm1d와 BatchNorm2d

PyTorch에서는 직접 구현하지 않고 이런걸 쓴다.

### Fully Connected Layer

```text
nn.BatchNorm1d(num_features)

예를 들어

nn.Linear(784, 256),
nn.BatchNorm1d(256),
nn.ReLU()
```

### CNN
```text
nn.BatchNorm2d(num_channels)

예를 들어

nn.Conv2d(
    in_channels=3,
    out_channels=64,
    kernel_size=3,
    padding=1
),

nn.BatchNorm2d(64), 
nn.ReLU()
```

nn.BatchNorm2d(64)는 이미지 크기가 아니라 Conv의 출력 채널 수이다.

## 8. 학습과 추론에서 동작이 다르다.

Batch Normalization에서 매우 중요한 부분이다.

### Training

학습 중에는 현재 minibatch의 평균과 분산을 사용한다.

현재 minibatch -> 평균, 분산 계산 -> BatchNorm

```text
Batch 1 → μ₁, σ₁ 
Batch 2 → μ₂, σ₂ 
Batch 3 → μ₃, σ₃
```
batch마다 조금씩 다른 값이 사용된다.

### Evaluation

실제 추론할 때는 이미지 한 장만 들어올 수도 있다.

사용자 이미지 한 장만 있는 상태에서 batch 평균을 계산하는 건 적절하지 않다. 그래서 학습하는 동안 `running_mean`, `running_var`를 계속 저장한다.

추론할 때 이 값을 사용한다.

Training -> 현재 batch의 평균/분산 사용 -> running mean/variance 업데이트  
Evaluation -> 저장된 running mean/variance 사용

그래서 model.train()과 model.eval() 차이가 중요하다. BatchNorm과 Dropout은 두 mode에서 동작이 달라지는 대표적인 layer이다.

## 9. Batch Normalization 구현

In [ ]:
def batch_norm(X, gamma, beta, eps=1e-5):

    mean = X.mean(dim=0) # 평균 계산

    var = ((X - mean) ** 2).mean(dim=0) # 분산 계산

    X_hat = (X - mean) / torch.sqrt(var + eps) # 정규화

    Y = gamma * X_hat + beta # 스케일 조정 및 이동

    return Y

## 10. LeNet에 Batch Normalization 추가하기

기존 LeNet이 

    Conv -> Sigmoid -> Pool 

였다면 Batch Norm을 추가해서

    Conv -> BatchNorm -> Sigmoid -> Pool

으로 만들 수 있다. 

In [ ]:
class BNLeNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            nn.Conv2d(1, 6, kernel_size=5), # conv
            nn.BatchNorm2d(6),              # batch norm 
            nn.Sigmoid(),                   # activation 반복 구조

            nn.AvgPool2d(
                kernel_size=2,
                stride=2
            ),

            nn.Conv2d(6, 16, kernel_size=5),
            nn.BatchNorm2d(16),
            nn.Sigmoid(),

            nn.AvgPool2d(
                kernel_size=2,
                stride=2
            ),

            nn.Flatten(),

            nn.Linear(16 * 4 * 4, 120),
            nn.BatchNorm1d(120),
            nn.Sigmoid(),

            nn.Linear(120, 84),
            nn.BatchNorm1d(84),
            nn.Sigmoid(),

            nn.Linear(84, 10)
        )

    def forward(self, X):
        return self.net(X)

## 11. BatchNorm이 학습에 도움을 주는 이유

BatchNorm을 사용하면 중간 layer의 값들이 지나치게 커지거나 작아지는 것을 어느정도 막을 수 있다.

예를 들어 BatchNorm이 없다면 학습 도중

```text
Layer 1 → 값 크기 0.01
Layer 2 → 값 크기 10
Layer 3 → 값 크기 1000
Layer 4 → 값 크기 0.0001
```

이렇게 scale이 크게 달라질 수 있다.

BatchNorm을 사용하면 이러한 intermediate activation을 계속 적절한 scale로 조절한다. 

일반적으로 이런 효과를 얻을 수 있다.

- 학습 안정성 증가
- 수렴 속도 개선
- 더 큰 learning rate 사용 가능

그리고 minibatch마다 평균과 분산이 조금씩 달라지므로 약간의 noise가 발생한다. 이것이 일종의 regularization 효과를 만들기도 한다.

## 12. Internal Covariate Shift에 대한 주의

Batch Normalization이 처음 발표됐을 때는 이렇게 설명했다.

> 학습 중 hidden layer의 입력 분포가 계속 변하는 Internal Covariate Shift를 줄여주기 때문에 성능이 좋아진다.

하지만 현재는 이게 BatchNorm의 효과를 충분히 설명하지 못한다고 알려져 있다. 실제로 중요한 건

    중간 activation의 scale 안정화 + 쉬워진 optimization + 일정한 regularization 효과

이렇게 이해하면 좋다.

## 13. Layer Normalization과의 차이

BatchNorm은 여러 sample을 묶은 batch를 이용해서 통계량을 계산한다.

반면 Layer Normalization은 각 sample 내부에서 정규화한다.

```text
Sample 1 → 자체 평균/분산
Sample 2 → 자체 평균/분산
Sample 3 → 자체 평균/분산
```

LayerNorm은 batch size에 영향을 받지 않는다.

`BatchNorm`: batch에 의존  
`LayerNorm`: 각 sample 기준, batch에 의존하지 않음

Transformer를 공부할 때 LayerNorm이 계속 등장하므로 이 차이는 기억해두는 것이 좋다.

## 14. 오늘의 정리

- Batch Normalization은 신경망 중간값을 정규화하는 방법이다.
- minibatch의 평균과 분산을 이용한다.
- 기본적으로 평균을 0, 분산을 1에 가깝게 만든다.
- 정규화 후 학습 가능한 $\gamma$와 $\beta$를 이용해 다시 scale과 위치를 조절한다.
- $\gamma$와 $\beta$ 역시 역전파를 통해 학습되는 parameter다.
- 일반적으로 Conv → BatchNorm → Activation 순서로 사용한다.
- CNN에서는 채널별로 Batch Normalization을 수행한다.
- BatchNorm2d(C)의 C는 Conv의 출력 채널 수다.
- 학습 시 현재 minibatch의 평균과 분산을 사용한다.
- 추론 시에는 학습 중 누적한 running mean과 running variance를 사용한다.
- 따라서 BatchNorm은 model.train()과 model.eval()에서 동작이 다르다.
- BatchNorm은 optimization을 안정시키고 수렴을 빠르게 하는 데 도움이 된다.
- minibatch 통계의 noise 때문에 regularization 효과도 어느 정도 발생한다.
- BatchNorm의 효과를 단순히 Internal Covariate Shift 때문이라고 설명하는 것은 정확하지 않다.
- LayerNorm은 batch가 아니라 각 sample을 기준으로 정규화하며 이후 Transformer에서 매우 중요하게 사용된다.